# Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np

# Step 2: Load Dataset


In [ ]:
df = pd.read_csv("/content/UpdatedResumeDataSet.csv")

# Step 3: Display First 5 Rows



In [ ]:


df.head()

# Step 4: Dataset Information

In [ ]:


print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

# Step 5: Check Job Categories

In [ ]:


print(df['Category'].value_counts())

# Step 6: Check Duplicate Resumes

In [ ]:


print("Duplicate Rows:", df.duplicated().sum())

# Install Required Libraries:

In [ ]:


!pip -q install nltk

Step 7: NLP Preprocessing

In [ ]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# Step 8: Text Preprocessing Function

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_resume(text):

    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'www\S+', ' ', text)
    text = re.sub(r'@\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    text = text.lower()

    words = word_tokenize(text)

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

# Step 9: Clean Resume Text

In [ ]:
df["Clean_Resume"] = df["Resume"].apply(clean_resume)

# Step 10: Display Cleaned Resume

In [ ]:
df[["Category","Clean_Resume"]].head()

Step 11: Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Category_Label"] = le.fit_transform(df["Category"])

print(le.classes_)

Step 12: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df["Clean_Resume"]
y = df["Category_Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Step 13: Tokenization


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

max_len = 200

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

print(X_train_pad.shape)
print(X_test_pad.shape)

Step 14: Build Deep Learning Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

vocab_size = 5000
num_classes = len(le.classes_)

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

Step 15: Train Model

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_test_pad, y_test),
    epochs=10,
    batch_size=32
)

Step 16: Padding

In [ ]:
max_length = 200

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post"
)

print(X_train_pad.shape)
print(X_test_pad.shape)

# Evaluate Model

In [ ]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy)

# Plot Accuracy

In [ ]:
# Plot Accuracy

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

Resume Prediction Function

In [ ]:
import numpy as np

def predict_resume(resume_text):

    cleaned = clean_resume(resume_text)

    seq = tokenizer.texts_to_sequences([cleaned])

    pad = pad_sequences(seq, maxlen=max_length, padding="post")

    prediction = model.predict(pad, verbose=0)

    label = np.argmax(prediction)

    return le.inverse_transform([label])[0]

Test Prediction

In [ ]:
sample_resume = """
Python, Machine Learning, Deep Learning,
SQL, Power BI, TensorFlow, Data Analysis,
Pandas, NumPy, Scikit-learn
"""

print("Predicted Role:", predict_resume(sample_resume))

Skill Extraction

In [ ]:
skills = [
    "Python","Java","SQL","Machine Learning","Deep Learning",
    "TensorFlow","PyTorch","Pandas","NumPy","Scikit-learn",
    "Power BI","Tableau","Excel","AWS","Docker","Kubernetes",
    "Flask","Django","FastAPI","Git","Linux","HTML","CSS","JavaScript"
]

def extract_skills(text):
    text = text.lower()
    found = []

    for skill in skills:
        if skill.lower() in text:
            found.append(skill)

    return sorted(set(found))

Test Skill Extraction

In [ ]:
print(extract_skills(sample_resume))

Resume Score

In [ ]:
required_skills = [
    "Python",
    "SQL",
    "Machine Learning",
    "Deep Learning",
    "Pandas",
    "NumPy",
    "Scikit-learn",
    "TensorFlow"
]

def resume_score(found_skills):
    score = (len(found_skills) / len(required_skills)) * 100
    return min(round(score, 2), 100)

In [ ]:
found = extract_skills(sample_resume)

score = resume_score(found)

print("Resume Score:", score, "%")

Missing Skills

In [ ]:
def missing_skills(found_skills):
    missing = []

    for skill in required_skills:
        if skill not in found_skills:
            missing.append(skill)

    return missing

In [ ]:
print("Missing Skills:")
print(missing_skills(found))

In [ ]:
def suggestions(missing):
    if len(missing) == 0:
        print("Excellent Resume! All important skills are present.")
    else:
        print("Recommended Skills to Learn:")
        for skill in missing:
            print("-", skill)

In [ ]:
suggestions(missing_skills(found))

Course Recommendation

In [ ]:
course_recommendations = {
    "Python": "Python for Everybody - Coursera",
    "SQL": "SQL for Data Science - Coursera",
    "Machine Learning": "Machine Learning by Andrew Ng",
    "Deep Learning": "Deep Learning Specialization - Coursera",
    "TensorFlow": "TensorFlow Developer Certificate",
    "Scikit-learn": "Scikit-learn Official Documentation",
    "Pandas": "Data Analysis with Pandas",
    "NumPy": "NumPy Official Tutorial"
}

In [ ]:
def recommend_courses(missing):

    print("Recommended Courses:\n")

    if len(missing) == 0:
        print("No recommendations needed.")
        return

    for skill in missing:
        if skill in course_recommendations:
            print(f"{skill} ➜ {course_recommendations[skill]}")

In [ ]:
recommend_courses(missing_skills(found))

In [ ]:
def analyze_resume(resume):

    print("="*50)

    role = predict_resume(resume)
    print("Predicted Role:", role)

    skills = extract_skills(resume)
    print("\nSkills Found:")
    print(skills)

    score = resume_score(skills)
    print("\nResume Score:", score, "%")

    missing = missing_skills(skills)

    print("\nMissing Skills:")
    print(missing)

    print("\nSuggestions:")
    suggestions(missing)

    print("\nCourse Recommendations:")
    recommend_courses(missing)

    print("="*50)

In [ ]:
analyze_resume(sample_resume)

Step 17: Install PDF Libraries

In [ ]:
!pip install pymupdf

Step 18: Import Libraries

In [ ]:
import fitz
from google.colab import files

Step 19: Upload Resume PDF

In [ ]:
uploaded = files.upload()

Step 20: Extract Text from Resume PDF

In [ ]:
filename = "sample_resume.pdf" # Using the filename known to exist from the file system list

# Construct the full path to the uploaded file
full_path = "/content/" + filename

doc = fitz.open(full_path)

resume_text = ""

for page in doc:
    resume_text += page.get_text()

doc.close()

print(resume_text[:2000])

Step 21: Clean Extracted Resume

In [ ]:
import re

def clean_uploaded_resume(text):
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower()

clean_resume = clean_uploaded_resume(resume_text)

print(clean_resume[:1000])

Step 22: Predict Resume Category using Deep Learning Model

In [ ]:
sequence = tokenizer.texts_to_sequences([clean_resume])

padded = pad_sequences(sequence,
                       maxlen=max_length,
                       padding="post",
                       truncating="post")

prediction = model.predict(padded)

predicted_class = prediction.argmax(axis=1)[0]

predicted_role = le.inverse_transform([predicted_class])[0]

print("Predicted Role :", predicted_role)

Step 23: Resume Score (AI Resume Score)

In [ ]:
skills = [
    "python","java","sql","machine learning","deep learning",
    "tensorflow","pytorch","pandas","numpy","scikit-learn",
    "power bi","tableau","excel","aws","docker",
    "kubernetes","git","linux","html","css","javascript",
    "flask","django","fastapi"
]

resume_lower = clean_resume.lower()

found_skills = []

for skill in skills:
    if skill.lower() in resume_lower:
        found_skills.append(skill)

score = min(len(found_skills) * 4, 100)

print("Resume Score:", score, "/100")

Step 24: Show Detected Skills

In [ ]:
print("Detected Skills:\n")

for skill in sorted(found_skills):
    print(skill)

Step 25: Find Missing Skills

In [ ]:
required_skills = [
    "python",
    "sql",
    "machine learning",
    "deep learning",
    "pandas",
    "numpy",
    "scikit-learn",
    "tensorflow",
    "power bi",
    "tableau",
    "git"
]

missing_skills = []

for skill in required_skills:
    if skill not in found_skills:
        missing_skills.append(skill)

print("Missing Skills:\n")

for skill in missing_skills:
    print(skill)

Step 26: AI Recommendations

In [ ]:
print(" AI Resume Analysis \n")

print("Predicted Role :", predicted_role)

print("\nResume Score :", score, "/100")

print("\nDetected Skills:")

for skill in sorted(found_skills):
    print( skill)

print("\nMissing Skills:")

for skill in missing_skills:
    print(skill)

print("\nRecommendations:")

if score >= 80:
    print("Excellent Resume. Ready for interviews.")
elif score >= 60:
    print("Good Resume. Improve missing skills.")
else:
    print("Resume needs significant improvement.")

Step 27: Recommend Learning Resources

In [ ]:
courses = {
    "python":"Python Programming",
    "sql":"SQL for Data Analysis",
    "machine learning":"Machine Learning Specialization",
    "deep learning":"Deep Learning Specialization",
    "tensorflow":"TensorFlow Developer Course",
    "power bi":"Microsoft Power BI",
    "tableau":"Tableau Data Visualization",
    "git":"Git & GitHub"
}

print("Recommended Courses:\n")

for skill in missing_skills:
    if skill in courses:
        print(f"{skill}  -->  {courses[skill]}")

Step 28: Resume Strength Analysis

In [ ]:
print(" Resume Strength \n")

if score >= 85:
    level = "Excellent"
elif score >= 70:
    level = "Very Good"
elif score >= 55:
    level = "Good"
elif score >= 40:
    level = "Average"
else:
    level = "Needs Improvement"

print("Resume Level :", level)

Step 29: ATS Compatibility Score

In [ ]:
ats_score = min(score + 5, 100)

print("ATS Compatibility Score :", ats_score, "/100")

if ats_score >= 80:
    print(" ATS Friendly Resume")
else:
    print(" Improve Resume Formatting and Skills")

Step 30: Final Professional Report

In [ ]:
print("="*60)
print("          AI RESUME ANALYZER REPORT")
print("="*60)

print(f"Predicted Role      : {predicted_role}")
print(f"Resume Score        : {score}/100")
print(f"ATS Score           : {ats_score}/100")
print(f"Skills Found        : {len(found_skills)}")
print(f"Missing Skills      : {len(missing_skills)}")

print("\nDetected Skills:")
for skill in found_skills:
    print( skill)

print("\nMissing Skills:")
for skill in missing_skills:
    print( skill)

print("\nOverall Recommendation:")

if score >= 80:
    print("Excellent Resume. Ready for interviews.")
elif score >= 60:
    print("Good Resume. Add missing skills.")
else:
    print("Resume needs improvement before applying.")

print("="*60)

Step 31: Save Deep Learning Model

In [ ]:
model.save("resume_classifier_model.h5")

print("Model Saved Successfully")

Step 32: Save Tokenizer

In [ ]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Tokenizer Saved Successfully")

Step 33: Save Label Encoder

In [ ]:
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Label Encoder Saved Successfully")

Step 34: Download Saved Files

In [ ]:
from google.colab import files

files.download("resume_classifier_model.h5")
files.download("tokenizer.pkl")
files.download("label_encoder.pkl")

Define Important Skills

In [ ]:
required_skills = [
    "Python",
    "SQL",
    "Machine Learning",
    "Deep Learning",
    "TensorFlow",
    "Scikit-learn",
    "Pandas",
    "NumPy",
    "Power BI",
    "Excel",
    "Git",
    "Docker",
    "AWS",
    "Linux"
]

ATS Score Function

In [ ]:
def calculate_ats_score(resume_text):

    resume_text = resume_text.lower()

    matched_skills = []

    for skill in required_skills:
        if skill.lower() in resume_text:
            matched_skills.append(skill)

    skill_score = (len(matched_skills) / len(required_skills)) * 60

    education_score = 10 if any(word in resume_text for word in ["b.tech","btech","mca","msc","bsc","be"]) else 0

    project_score = 10 if "project" in resume_text else 0

    experience_score = 10 if any(word in resume_text for word in ["experience","internship","worked"]) else 0

    certification_score = 10 if any(word in resume_text for word in ["certificate","certification"]) else 0

    total_score = skill_score + education_score + project_score + experience_score + certification_score

    return round(total_score,2), matched_skills

In [ ]:
ats_score, matched = calculate_ats_score(resume_text)

print("ATS Resume Score:", ats_score)

print()

print("Matched Skills:")

print(matched)

Step 37: Resume Grade

In [ ]:
def resume_grade(score):

    if score >= 90:
        return "A+ (Excellent Resume)"

    elif score >= 80:
        return "A (Very Strong)"

    elif score >= 70:
        return "B (Good)"

    elif score >= 60:
        return "C (Needs Improvement)"

    else:
        return "D (Poor Resume)"

In [ ]:
print("Resume Grade :", resume_grade(ats_score))

Step 38: Job Description Matching

In [ ]:
job_description = """
We are looking for a Data Scientist with experience in Python,
Machine Learning, Deep Learning, SQL, Pandas,
NumPy, TensorFlow, Scikit-learn, Git, Docker and AWS.
"""

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
documents = [clean_resume, job_description.lower()]

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(documents)

similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

match_percentage = round(similarity * 100, 2)

print("Resume Match Percentage :", match_percentage, "%")

Step 39: Match Quality

In [ ]:
if match_percentage >= 85:
    print("Excellent Match ")

elif match_percentage >= 70:
    print("Good Match ")

elif match_percentage >= 50:
    print("Average Match ")

else:
    print("Low Match ")

Step 40: Missing Skills from Job Description

In [ ]:
job_skills = [
    "Python",
    "SQL",
    "Machine Learning",
    "Deep Learning",
    "TensorFlow",
    "Scikit-learn",
    "Pandas",
    "NumPy",
    "Git",
    "Docker",
    "AWS"
]

missing = []

for skill in job_skills:
    if skill.lower() not in clean_resume:
        missing.append(skill)

print("Skills Required for Job:\n")

for skill in missing:
    print(skill)

Step 41: AI Suggestions

In [ ]:
print("AI Suggestions \n")

if len(missing) == 0:
    print("✅ Your resume matches the job description very well.")
else:
    print("To improve your chances, consider adding or strengthening these skills:")

    for skill in missing:
        print("•", skill)

Step 42: Prediction Confidence Score

In [ ]:
import numpy as np

prediction = model.predict(padded)

confidence = np.max(prediction) * 100

predicted_class = np.argmax(prediction)

predicted_role = le.inverse_transform([predicted_class])[0]

print("Predicted Role :", predicted_role)
print("Confidence Score :", round(confidence,2), "%")

Step 43: Confidence Level

In [ ]:
if confidence >= 90:
    print("🟢 Very High Confidence")
elif confidence >= 75:
    print("🟡 High Confidence")
elif confidence >= 60:
    print("🟠 Medium Confidence")
else:
    print("🔴 Low Confidence")

Step 44: Final AI Resume Summary

In [ ]:
print("="*60)
print("          AI RESUME ANALYZER REPORT")
print("="*60)

print(f"Predicted Role      : {predicted_role}")
print(f"Confidence Score    : {round(confidence,2)} %")
print(f"ATS Score           : {ats_score}/100")
print(f"Resume Grade        : {resume_grade(ats_score)}")
print(f"Job Match           : {match_percentage}%")

print("\nDetected Skills:")
for skill in matched:
    print("✔", skill)

print("\nMissing Skills:")
for skill in missing:
    print( skill)

print("\nRecommendations:")
if len(missing) == 0:
    print("Excellent! Your resume is well aligned with the job description.")
else:
    print("Focus on learning the missing skills listed above.")

Step 45: ATS Score Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5,5))

plt.bar(["ATS Score"], [ats_score])

plt.ylim(0,100)

plt.title("ATS Resume Score")

plt.ylabel("Score")

plt.show()

Step 46: Resume vs Job Match Visualization

In [ ]:
plt.figure(figsize=(5,5))

plt.bar(["Resume Match"], [match_percentage])

plt.ylim(0,100)

plt.title("Resume vs Job Description Match")

plt.ylabel("Match %")

plt.show()

Step 47: Skills Found vs Missing Skills

In [ ]:
labels = ["Skills Found","Missing Skills"]

values = [len(matched), len(missing)]

plt.figure(figsize=(5,5))

plt.bar(labels, values)

plt.title("Skills Analysis")

plt.ylabel("Count")

plt.show()

Step 48: Pie Chart

In [ ]:
labels = ["Matched","Missing"]

sizes = [len(matched), len(missing)]

plt.figure(figsize=(6,6))

plt.pie(sizes,
        labels=labels,
        autopct="%1.1f%%",
        startangle=90)

plt.title("Resume Skill Distribution")

plt.show()

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate(clean_resume)

plt.figure(figsize=(12,6))

plt.imshow(wordcloud)

plt.axis("off")

plt.title("Resume Word Cloud")

plt.show()

In [ ]:
# Step 50: Final AI Resume Dashboard

print("="*70)
print("          AI RESUME ANALYZER DASHBOARD")
print("="*70)

print(f"📄 Predicted Role        : {predicted_role}")
print(f"🎯 Confidence Score      : {round(confidence,2)} %")
print(f"📊 ATS Score             : {ats_score}/100")
print(f"🏆 Resume Grade          : {resume_grade(ats_score)}")
print(f"💼 Job Match             : {match_percentage}%")

print("\n" + "="*70)

print("✅ Skills Found")
for skill in matched:
    print("   ✔", skill)

print("\n❌ Missing Skills")
for skill in missing:
    print("   ✘", skill)

print("\n💡 Recommendations")

if len(missing)==0:
    print("Your resume is well optimized.")
else:
    for skill in missing:
        print(f"Learn {skill}")

print("="*70)

Step 51: Strengths & Weaknesses

In [ ]:
print("Resume Analysis \n")

print("Strengths")

if len(matched) > 0:
    for skill in matched:
        print( skill)

print("\nWeaknesses")

if len(missing) > 0:
    for skill in missing:
        print( skill)
else:
    print("No major weaknesses detected.")

Step 52: Final Hiring Recommendation

In [ ]:
print("\nHiring Recommendation")

if ats_score >= 90:
    print("Highly Recommended")

elif ats_score >= 80:
    print("Recommended")

elif ats_score >= 65:
    print("Recommended after Skill Improvement")

else:
    print("Not Recommended Currently")

In [ ]:
report = f"""
===============================
        AI RESUME REPORT
===============================

Predicted Role : {predicted_role}

Confidence Score : {round(confidence,2)} %

ATS Score : {ats_score}/100

Resume Grade : {resume_grade(ats_score)}

Job Match : {match_percentage} %

Detected Skills

"""

for skill in matched:
    report += f" {skill}\n"

report += "\nMissing Skills\n\n"

for skill in missing:
    report += f" {skill}\n"

report += "\nRecommendations\n\n"

if len(missing)==0:
    report += "Resume is well optimized.\n"
else:
    for skill in missing:
        report += f"Learn {skill}\n"

print(report)

In [ ]:
with open("Resume_Analysis_Report.txt", "w") as file:
    file.write(report)

print("Report Saved Successfully")

In [ ]:
improvement = len(missing) * 5

future_score = min(100, ats_score + improvement)

print("Current ATS Score :", ats_score)
print("Potential Score After Improvements :", future_score)

In [ ]:
career_advice = {
    "Data Science":
        "Build ML and Deep Learning projects, strengthen SQL and Statistics.",

    "Web Designing":
        "Learn React, Node.js and build responsive web applications.",

    "Java Developer":
        "Practice Spring Boot, REST APIs and Data Structures.",

    "Python Developer":
        "Learn Django/Flask, APIs and deployment.",

    "DevOps Engineer":
        "Learn Docker, Kubernetes, AWS and CI/CD."
}

print("========== Career Advice ==========\n")

if predicted_role in career_advice:
    print(career_advice[predicted_role])
else:
    print("Continue improving your technical skills and build more real-world projects.")

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
model_st = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
resume_embedding = model_st.encode(clean_resume)

jd_embedding = model_st.encode(job_description)

semantic_similarity = cosine_similarity(
    [resume_embedding],
    [jd_embedding]
)[0][0]

print("Semantic Resume Match Score:", round(semantic_similarity * 100, 2), "%")

In [ ]:
skills = [
    "Python",
    "SQL",
    "Machine Learning",
    "Deep Learning",
    "TensorFlow",
    "Scikit-learn",
    "Pandas",
    "NumPy",
    "Docker",
    "AWS",
    "Git",
    "Power BI"
]

resume_embedding = model_st.encode(clean_resume)

semantic_skills = []

for skill in skills:
    skill_embedding = model_st.encode(skill)

    similarity = cosine_similarity(
        [resume_embedding],
        [skill_embedding]
    )[0][0]

    if similarity >= 0.25:
        semantic_skills.append((skill, round(similarity * 100, 2)))

print("Semantic Skills Found\n")

for skill, score in semantic_skills:
    print(f"{skill}  -->  {score}%")

In [ ]:
semantic_skills = sorted(
    semantic_skills,
    key=lambda x: x[1],
    reverse=True
)

print("Top 5 Matching Skills\n")

for skill, score in semantic_skills[:5]:
    print(f"{skill} : {score}%")

In [ ]:
if ats_score >= 90 and semantic_similarity >= 0.80:
    decision = "Highly Recommended"

elif ats_score >= 75:
    decision = "Shortlist"

else:
    decision = "Needs Improvement"

print("Recruiter Decision :", decision)

In [ ]:
print("Why was this resume classified as", predicted_role, "?")

print("\nReason:")

for skill, score in semantic_skills[:5]:
    print(f" {skill} contributed with semantic relevance ({score}%)")